# P04 — UK House Price Dynamics
## Notebook 02: Analysis

This notebook performs the core statistical analysis for P04. It loads the
analysis-ready dataset produced in notebook 01 and addresses four questions:

1. Which English local authorities are most and least affordable in 2024,
   and how does affordability vary by region?
2. Is London statistically significantly less affordable than the rest of
   England, and by how much?
3. Does deprivation predict unaffordability at local authority level?
4. What does an OLS regression of affordability on deprivation reveal about
   the relationship when region is controlled for?

The affordability ratio throughout is defined as median house price divided
by median gross annual earnings. A ratio of 10.0 means the median home costs
ten times the median annual salary in that local authority.

## Section 1 — Environment Setup

We verify the working directory, import all libraries and load the
analysis-ready dataset exported by notebook 01.

In [1]:
import os
import sys

os.getcwd()

'C:\\Users\\TEST\\OneDrive\\Documents\\The United Kingdom\\Jobs\\Data Science\\portfolio\\project4\\notebooks'

In [2]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import warnings

warnings.filterwarnings("ignore")

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
import config

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Statsmodels version:", sm.__version__)

Python version: 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
Pandas version: 3.0.2
NumPy version: 2.4.4
Statsmodels version: 0.14.6


## Section 2 — Load Data

We load house_prices_final.csv produced by notebook 01 and the IMD 2019
dataset from P03. The IMD file is the same one used in P02 and P03 and
is stored in data/raw/ alongside the other raw inputs.

We merge the IMD score onto the main dataset on lad_code so that all
variables needed for the regression are in a single DataFrame.

In [3]:
# Load main dataset
df = pd.read_csv(config.HOUSE_PRICES_FINAL)

print(f"Main dataset: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nAffordability ratio summary:")
print(df["affordability_ratio"].describe().round(2))

Main dataset: (296, 11)
Columns: ['lad_code', 'transaction_count', 'median_price_ppd', 'mean_price_ppd', 'lower_quartile_ppd', 'lad_name', 'region_code', 'region_name', 'median_price_ons', 'median_earnings', 'affordability_ratio']

Affordability ratio summary:
count    295.00
mean      10.34
std        3.67
min        4.57
25%        7.62
50%        9.94
75%       12.37
max       32.04
Name: affordability_ratio, dtype: float64


## Section 3 — Load and Merge IMD 2019

The Index of Multiple Deprivation 2019 (IMD) measures relative deprivation
across English local authorities. We used this dataset in both P02 and P03.
Here we reuse it to test whether deprived areas are also unaffordable areas.

The IMD file contains scores at LSOA level. We aggregate to LA level by
taking the mean IMD score across all LSOAs within each LA, weighted by
the LSOA population. This gives a population-weighted mean deprivation
score for each local authority - the same approach used in P03.

In [5]:
# Load IMD 2019
imd_raw = pd.read_csv(config.IMD_FILE, encoding="latin-1")

print(f"IMD raw shape: {imd_raw.shape}")
print(f"Columns: {imd_raw.columns.tolist()[:8]}")
imd_raw.head(3)

IMD raw shape: (32844, 57)
Columns: ['LSOA code (2011)', 'LSOA name (2011)', 'Local Authority District code (2019)', 'Local Authority District name (2019)', 'Index of Multiple Deprivation (IMD) Score', 'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)', 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)', 'Income Score (rate)']


,LSOA code (2011),LSOA name (2011),Local Authority District code (2019),Local Authority District name (2019),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2015 (excluding prisoners),Dependent Children aged 0-15: mid 2015 (excluding prisoners),Population aged 16-59: mid 2015 (excluding prisoners),Older population aged 60 and over: mid 2015 (excluding prisoners),Working age population 18-59/64: for use with Employment Deprivation Domain (excluding prisoners)
0,E01000001,City of London 001A,E09000001,City of London,6.208,29199,9,0.007,32831,10,...,16364,5,1.503,1615,1,1296,175,656,465,715
1,E01000002,City of London 001B,E09000001,City of London,5.143,30379,10,0.034,29901,10,...,22676,7,1.196,2969,1,1156,182,580,394,620
2,E01000003,City of London 001C,E09000001,City of London,19.402,14915,5,0.086,18510,6,...,17318,6,2.207,162,1,1350,146,759,445,804


In [6]:
# Extract columns needed for population-weighted LA-level IMD
imd = imd_raw[[
    "LSOA code (2011)",
    "Local Authority District code (2019)",
    "Local Authority District name (2019)",
    "Index of Multiple Deprivation (IMD) Score",
    "Total population: mid 2015 (excluding prisoners)"
]].copy()

imd.columns = [
    "lsoa_code", "lad_code", "lad_name_imd",
    "imd_score", "population"
]

# Compute population-weighted mean IMD score per LA
imd_la = (
    imd.groupby("lad_code")
    .apply(
        lambda x: np.average(
            x["imd_score"],
            weights=x["population"]
        )
    )
    .reset_index()
)

imd_la.columns = ["lad_code", "imd_score_wtd"]

print(f"IMD aggregated to {imd_la.shape[0]} local authorities")
print(f"\nIMD score summary:")
print(imd_la["imd_score_wtd"].describe().round(2))

IMD aggregated to 317 local authorities

IMD score summary:
count    317.00
mean      19.62
std        7.98
min        5.54
25%       13.18
50%       18.22
75%       25.43
max       45.04
Name: imd_score_wtd, dtype: float64


## Explanation of the Results Above

This computes a population-weighted mean IMD score for each LA. A simple unweighted mean would give equal weight to every LSOA regardless of how many people live there. A deprived LSOA with 800 residents should contribute less to the LA average than a deprived LSOA with 2,400 residents. Weighting by population ensures the LA-level score reflects where people actually live, not just the number of geographic units.

317 local authorities -- the IMD uses 2019 LAD boundaries, which had 317 English LAs. Our house price data uses 2024/2025 boundaries, which have 296 LAs. The difference of 21 reflects local government reorganisations between 2019 and 2024 where multiple smaller districts were merged into new unitary authorities. When we merge the two datasets, some 2019 codes will not match any 2024 code and will be lost. We will check the merge rate next.

IMD score range 5.54 to 45.04 -- higher scores mean more deprived. The maximum of 45.04 is almost certainly a northern urban LA. The minimum of 5.54 is a wealthy rural or southern LA. The median of 18.22 and mean of 19.62 are close, suggesting a roughly symmetric distribution with slight right skew from the most deprived LAs.

In [7]:
# Merge IMD onto main dataset
df = df.merge(imd_la, on="lad_code", how="left")

matched   = df["imd_score_wtd"].notna().sum()
unmatched = df["imd_score_wtd"].isna().sum()

print(f"LAs matched to IMD:   {matched}")
print(f"LAs unmatched:        {unmatched}")

# Show unmatched LAs
if unmatched > 0:
    print(f"\nUnmatched LAs:")
    print(df[df["imd_score_wtd"].isna()][
        ["lad_code", "lad_name", "region_name", "median_price_ppd"]
    ])

LAs matched to IMD:   287
LAs unmatched:        9

Unmatched LAs:
      lad_code                 lad_name               region_name  \
56   E06000060          Buckinghamshire                South East   
57   E06000061   North Northamptonshire             East Midlands   
58   E06000062    West Northamptonshire             East Midlands   
59   E06000063               Cumberland                North West   
60   E06000064  Westmorland and Furness                North West   
61   E06000065          North Yorkshire  Yorkshire and The Humber   
62   E06000066                 Somerset                South West   
261  E08000038                 Barnsley  Yorkshire and The Humber   
262  E08000039                Sheffield  Yorkshire and The Humber   

     median_price_ppd  
56           450000.0  
57           265000.0  
58           300000.0  
59           175000.0  
60           240000.0  
61           270000.0  
62           277000.0  
261          175000.0  
262          213875.0  


In [8]:
# Predecessor LA codes for each reorganised LA
predecessor_map = {
    "E06000060": ["E07000004", "E07000005", "E07000006", "E07000007"],
    "E06000061": ["E07000150", "E07000152", "E07000153", "E07000158"],
    "E06000062": ["E07000151", "E07000154", "E07000155"],
    "E06000063": ["E07000026", "E07000028", "E07000029"],
    "E06000064": ["E07000027", "E07000030", "E07000031"],
    "E06000065": ["E07000163", "E07000164", "E07000165",
                  "E07000166", "E07000167", "E07000168", "E07000169"],
    "E06000066": ["E07000187", "E07000188", "E07000246", "E07000189"],
    "E08000038": ["E08000016"],
    "E08000039": ["E08000019"],
}

# For each new LA, compute population-weighted mean IMD from predecessors
for new_code, old_codes in predecessor_map.items():
    predecessor_lsoas = imd[imd["lad_code"].isin(old_codes)]
    if len(predecessor_lsoas) == 0:
        print(f"WARNING: no LSOAs found for {new_code}")
        continue
    wtd_imd = np.average(
        predecessor_lsoas["imd_score"],
        weights=predecessor_lsoas["population"]
    )
    df.loc[df["lad_code"] == new_code, "imd_score_wtd"] = wtd_imd

# Verify
remaining_missing = df["imd_score_wtd"].isna().sum()
print(f"Remaining unmatched after fix: {remaining_missing}")
print(f"\nFixed LAs:")
print(df[df["lad_code"].isin(predecessor_map.keys())][
    ["lad_code", "lad_name", "imd_score_wtd"]
].round(2))

Remaining unmatched after fix: 0

Fixed LAs:
      lad_code                 lad_name  imd_score_wtd
56   E06000060          Buckinghamshire          10.05
57   E06000061   North Northamptonshire          19.04
58   E06000062    West Northamptonshire          17.71
59   E06000063               Cumberland          23.09
60   E06000064  Westmorland and Furness          19.02
61   E06000065          North Yorkshire          14.76
62   E06000066                 Somerset          18.55
261  E08000038                 Barnsley          29.93
262  E08000039                Sheffield          27.06


### Explanation of the Results above

Predecessor mapping -- each new LA was created by merging two or more predecessor LAs. The IMD 2019 has scores at LSOA level, and each LSOA carries its old 2019 LAD code. By filtering LSOAs to the predecessor codes and computing a population-weighted average, we reconstruct what the IMD score would have been for the new LA boundary. This is the methodologically correct approach -- the same one used by ONS when they backcast statistics to new boundaries.

Barnsley and Sheffield -- these two had a different situation. They were not mergers of multiple LAs but single LAs that simply received new codes due to minor boundary adjustments. Their predecessor codes E08000016 and E08000019 map directly to all their LSOAs, so the weighted average is effectively the same as the original LA score.

Zero remaining unmatched -- all 296 LAs now have an IMD score. The dataset is complete for the regression analysis.

IMD scores look credible -- Buckinghamshire 10.05 is a wealthy Home Counties LA, correctly low. Barnsley 29.93 and Sheffield 27.06 are South Yorkshire industrial LAs, correctly high. Cumberland 23.09 reflects a mixed rural and post-industrial area in Cumbria. These values pass the sense check.

## Section 4 — Affordability Ranking

We rank all 296 English local authorities by affordability ratio and
identify the ten least affordable and ten most affordable areas. This
directly addresses the first research question: which areas of England
are most and least affordable in 2024?

In [9]:
# Drop the one LA missing affordability ratio before ranking
df_ranked = df.dropna(subset=["affordability_ratio"]).copy()

# Sort by affordability ratio descending (least affordable first)
df_ranked = df_ranked.sort_values("affordability_ratio", ascending=False)

print("Ten least affordable local authorities (2024):")
print(df_ranked[["lad_name", "region_name", "median_price_ppd",
                  "median_earnings", "affordability_ratio"]].head(10).to_string())

print("\nTen most affordable local authorities (2024):")
print(df_ranked[["lad_name", "region_name", "median_price_ppd",
                  "median_earnings", "affordability_ratio"]].tail(10).to_string())

Ten least affordable local authorities (2024):
                   lad_name      region_name  median_price_ppd  median_earnings  affordability_ratio
282  Kensington and Chelsea           London         1235000.0          38541.0            32.043798
119                 Dacorum  East of England          450000.0          18675.0            24.096386
192               Elmbridge       South East          655000.0          30000.0            21.833333
276                Haringey           London          563750.0          27716.0            20.340237
289    Richmond upon Thames           London          715000.0          35263.0            20.276210
295             Westminster           London          950000.0          49973.0            19.010266
265                  Barnet           London          580000.0          30926.0            18.754446
221               St Albans  East of England          625000.0          33468.0            18.674555
275  Hammersmith and Fulham           London

### Explanation of the Results above

Ten least affordable -- Kensington and Chelsea tops the list at 32.04, meaning the median home costs 32 times the median annual salary. This is an extreme outlier driven by both the highest median price in England at £1,235,000 and relatively modest workplace earnings of £38,541. Seven of the ten least affordable LAs are in London, with Dacorum and St Albans in the East of England and Elmbridge in the South East rounding out the list. Dacorum is a striking case -- its ratio of 24.10 is driven not by exceptionally high prices but by the lowest median earnings in the entire dataset at £18,675, which is almost certainly a suppression or data quality issue worth noting.

Ten most affordable -- all are northern or Midlands LAs. Kingston upon Hull is the most affordable at 4.57. Burnley, Hyndburn and County Durham follow closely. The pattern is clear -- low house prices combined with reasonable earnings produces the most affordable ratios. Blackpool at £135,000 median price is the second cheapest market in the dataset.

The North-South divide is stark -- the least affordable LA (Kensington and Chelsea, ratio 32.04) is seven times less affordable than the most affordable (Hull, ratio 4.57). This is the central finding of P04.

## Section 5 — Regional Analysis and Welch's t-test

We compute the mean affordability ratio by region and test whether London
is statistically significantly less affordable than the rest of England.

We use Welch's t-test rather than Student's t-test because we cannot assume
equal variances between London (33 LAs) and the rest of England (262 LAs).
The test statistic is:

    t = (AR_L - AR_R) / sqrt(s_L^2/n_L + s_R^2/n_R)

Where AR_L is the mean affordability ratio for London LAs, AR_R for the
rest of England, s^2 is the sample variance and n is the group size.
A p-value below 0.05 allows us to reject the null hypothesis that London
and the rest of England have the same mean affordability ratio.

In [10]:
# Regional summary
regional_summary = (
    df_ranked.groupby("region_name")
    .agg(
        la_count=("lad_code", "count"),
        mean_affordability=("affordability_ratio", "mean"),
        median_affordability=("affordability_ratio", "median"),
        mean_price=("median_price_ppd", "mean"),
        mean_earnings=("median_earnings", "mean")
    )
    .round(2)
    .sort_values("mean_affordability", ascending=False)
)

print("Regional affordability summary:")
print(regional_summary.to_string())

# Welch's t-test: London vs rest of England
london      = df_ranked[df_ranked["region_name"] == "London"]["affordability_ratio"]
rest        = df_ranked[df_ranked["region_name"] != "London"]["affordability_ratio"]

t_stat, p_value = stats.ttest_ind(london, rest, equal_var=False)
cohen_d = (london.mean() - rest.mean()) / np.sqrt(
    (london.std()**2 + rest.std()**2) / 2
)

print(f"\nWelch's t-test: London vs Rest of England")
print(f"London mean ratio:      {london.mean():.3f}  (n={len(london)})")
print(f"Rest of England mean:   {rest.mean():.3f}  (n={len(rest)})")
print(f"t-statistic:            {t_stat:.3f}")
print(f"p-value:                {p_value:.6f}")
print(f"Cohen's d:              {cohen_d:.3f}")

Regional affordability summary:
                          la_count  mean_affordability  median_affordability  mean_price  mean_earnings
region_name                                                                                            
London                          33               15.59                 15.01   580191.29       37856.45
South East                      64               12.05                 11.90   394672.12       32917.47
East of England                 45               11.76                 10.81   363996.24       31278.00
South West                      26               10.34                 10.47   313898.08       30519.12
West Midlands                   30                8.50                  8.30   260145.95       30773.90
East Midlands                   35                8.46                  8.23   248214.71       29514.57
North West                      35                6.94                  6.90   212401.93       30643.03
Yorkshire and The Humber        

Regional summary -- we group by region and compute four statistics per region: LA count, mean and median affordability ratio, mean house price and mean earnings. Sorting by mean affordability descending gives an immediate ranking of regions from least to most affordable. London sits at the top at 15.59, more than three points above the South East at 12.05. The North East at 5.90 is the most affordable region -- less than half as unaffordable as London.

Mean vs median affordability by region -- in London the mean (15.59) is notably higher than the median (15.01), confirming that a small number of extreme LAs like Kensington and Chelsea and Westminster pull the London mean upward. In the South West the mean (10.34) is slightly below the median (10.47), suggesting mild left skew -- a few relatively affordable rural LAs pulling the mean down.

Welch's t-test result -- t = 8.045, p < 0.000001, Cohen's d = 1.645. This is an exceptionally strong result. The p-value is so small it rounds to zero at six decimal places, meaning we can reject the null hypothesis with overwhelming confidence. London is statistically significantly less affordable than the rest of England.

Cohen's d = 1.645 -- this is a very large effect size by conventional benchmarks (small = 0.2, medium = 0.5, large = 0.8). A Cohen's d of 1.645 means the London and rest-of-England affordability distributions are separated by 1.645 standard deviations. For context, P02 found a Cohen's d of 1.975 for the North-South health divide. The housing affordability divide between London and England is nearly as large as the health divide -- a striking parallel finding across the portfolio.

The gap in absolute terms -- London mean ratio 15.595 minus rest of England 9.675 equals a gap of 5.92 ratio points. A Londoner needs almost six additional years of salary to afford the median home compared to the average English LA outside London.

## Section 6 — Deprivation and Affordability: Pearson Correlation

We test whether deprived local authorities are also unaffordable ones.
The hypothesis is that deprivation and unaffordability do not simply
co-locate -- in fact we expect the relationship to be complex. Highly
deprived areas tend to have low house prices, which could make them
appear affordable despite poor economic conditions. The Pearson
correlation will tell us the direction and strength of this relationship.

The Pearson correlation coefficient is:

    r = sum((AR_i - AR_mean)(IMD_i - IMD_mean)) /
        sqrt(sum((AR_i - AR_mean)^2) * sum((IMD_i - IMD_mean)^2))

Where AR_i is the affordability ratio and IMD_i is the IMD score for
local authority i.

In [11]:
# Drop any remaining NaN in affordability or IMD before correlation
df_corr = df_ranked.dropna(subset=["affordability_ratio", "imd_score_wtd"]).copy()

r, p_value = stats.pearsonr(
    df_corr["affordability_ratio"],
    df_corr["imd_score_wtd"]
)

print(f"Pearson correlation: affordability ratio vs IMD score")
print(f"r  = {r:.3f}")
print(f"R2 = {r**2:.3f}")
print(f"p  = {p_value:.6f}")
print(f"n  = {len(df_corr)}")

# Regional breakdown of correlation
print(f"\nCorrelation by region:")
for region in sorted(df_corr["region_name"].unique()):
    subset = df_corr[df_corr["region_name"] == region]
    if len(subset) > 5:
        r_reg, p_reg = stats.pearsonr(
            subset["affordability_ratio"],
            subset["imd_score_wtd"]
        )
        print(f"  {region:<35} r = {r_reg:6.3f}  n = {len(subset)}")

Pearson correlation: affordability ratio vs IMD score
r  = -0.509
R2 = 0.260
p  = 0.000000
n  = 295

Correlation by region:
  East Midlands                       r = -0.731  n = 35
  East of England                     r = -0.546  n = 45
  London                              r = -0.212  n = 33
  North East                          r = -0.572  n = 12
  North West                          r = -0.559  n = 35
  South East                          r = -0.608  n = 64
  South West                          r = -0.512  n = 26
  West Midlands                       r = -0.712  n = 30
  Yorkshire and The Humber            r = -0.884  n = 15


### Explanation of the Results above

stats.pearsonr() -- computes the Pearson correlation coefficient between affordability ratio and IMD score across all 295 LAs with complete data. It returns two values: r (the correlation coefficient) and the p-value for the null hypothesis that the true correlation is zero.

r = -0.509, R2 = 0.260, p < 0.000001 -- the negative sign is the key finding. More deprived areas (higher IMD score) are associated with lower affordability ratios -- meaning more deprived areas are actually more affordable by the price-to-earnings measure. This is the paradox at the heart of the deprivation-housing relationship: deprivation suppresses house prices faster than it suppresses earnings, producing lower affordability ratios in the most deprived areas. R2 = 0.260 means deprivation alone explains 26% of the variance in affordability ratios across English LAs.

Contrast with P02 and P03 -- in P02, deprivation correlated with life expectancy at r = -0.862. In P03, deprivation correlated with crime at r = +0.711. Here r = -0.509. The housing relationship is weaker and in the opposite direction to what a naive hypothesis might predict. This is a genuinely interesting finding -- deprivation, health and crime all cluster together, but housing affordability runs counter to that pattern at the national level.

Regional correlations -- Yorkshire and The Humber shows the strongest relationship at r = -0.884, meaning within that region deprived LAs are strongly more affordable. East Midlands (-0.731) and West Midlands (-0.712) follow. London shows the weakest relationship at r = -0.212 -- within London, deprivation and affordability are barely correlated because even relatively deprived London boroughs have very high house prices driven by the city-wide premium. This within-region analysis adds important nuance to the national figure.

## Section 7 — OLS Regression: Affordability on Deprivation

We fit an Ordinary Least Squares regression of affordability ratio on
IMD score using statsmodels. This is the first formal Era 2 library use
in the portfolio.

The model is:

    AR_i = B0 + B1 * IMD_i + e_i

Where AR_i is the affordability ratio for local authority i, IMD_i is
the population-weighted mean IMD score, B0 is the intercept, B1 is the
slope coefficient and e_i is the residual.

B1 tells us the change in affordability ratio for each one-unit increase
in IMD score. A negative B1 confirms that more deprived areas have lower
affordability ratios -- consistent with the Pearson correlation above.

We add a constant to the model explicitly using sm.add_constant(), which
is required by statsmodels. Unlike scikit-learn, statsmodels does not add
an intercept automatically.

In [12]:
# Prepare regression variables
X = sm.add_constant(df_corr["imd_score_wtd"])
y = df_corr["affordability_ratio"]

# Fit OLS model
model = sm.OLS(y, X).fit()

print(model.summary())

                             OLS Regression Results                            
Dep. Variable:     affordability_ratio   R-squared:                       0.260
Model:                             OLS   Adj. R-squared:                  0.257
Method:                  Least Squares   F-statistic:                     102.7
Date:                 Sun, 12 Apr 2026   Prob (F-statistic):           6.87e-21
Time:                         14:25:22   Log-Likelihood:                -757.46
No. Observations:                  295   AIC:                             1519.
Df Residuals:                      293   BIC:                             1526.
Df Model:                            1                                         
Covariance Type:             nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            14.9686      0.493 

### Explanation of the Results above

Key Results:
R-squared = 0.260 -- IMD score alone explains 26.0% of the variance in affordability ratios across English LAs. The adjusted R-squared of 0.257 is almost identical, confirming that adding one predictor to 295 observations does not overfit.

B0 (intercept) = 14.969 -- when IMD score is zero (a hypothetical LA with no deprivation at all) the predicted affordability ratio would be 14.97. This is not directly interpretable as a real-world value since no LA has an IMD score of zero, but it anchors the regression line.

B1 (slope) = -0.233, p < 0.001, 95% CI [-0.278, -0.188] -- for every one-unit increase in IMD score, the affordability ratio falls by 0.233. The confidence interval does not include zero, confirming the effect is statistically significant. A ten-point increase in IMD score -- roughly the difference between a moderately deprived and a highly deprived LA -- is associated with an affordability ratio 2.33 points lower.

F-statistic = 102.7, p = 6.87e-21 -- the overall model is highly statistically significant. The probability that the observed relationship arose by chance is essentially zero.

Omnibus and Jarque-Bera tests -- both flag significant departures from normality in the residuals (skew = 2.124, kurtosis = 11.830). This is driven by the extreme outliers -- Kensington and Chelsea, Dacorum -- pulling the residual distribution rightward. The OLS coefficient estimates remain unbiased despite non-normal residuals, but standard errors may be slightly underestimated. For a portfolio project at this level this is noted and acceptable. 
In a production model we would consider robust standard errors or a log transformation of the affordability ratio.

Durbin-Watson = 0.486 -- values close to 2.0 indicate no autocorrelation. At 0.486 this suggests positive spatial autocorrelation -- nearby LAs tend to have similar affordability ratios, which makes geographic sense. 

## Section 8 — Summary of Findings

| Finding | Result |
|---|---|
| Most unaffordable LA | Kensington and Chelsea: ratio 32.04 |
| Most affordable LA | Kingston upon Hull: ratio 4.57 |
| London vs Rest of England | t = 8.045, Cohen's d = 1.645, p < 0.001 |
| Deprivation vs affordability | r = -0.509, R2 = 0.260, p < 0.001 |
| OLS slope | -0.233 per IMD point (95% CI: -0.278 to -0.188) |

The central paradox of P04: deprived areas are more affordable by the
price-to-earnings measure because deprivation suppresses house prices
faster than it suppresses earnings. Affordability and deprivation are
not the same problem, though they share geography in complex ways.

In [13]:
# Re-export with IMD score included
df.to_csv(config.HOUSE_PRICES_FINAL, index=False)
print(f"Re-exported: {df.shape}")
print(df.columns.tolist())

Re-exported: (296, 12)
['lad_code', 'transaction_count', 'median_price_ppd', 'mean_price_ppd', 'lower_quartile_ppd', 'lad_name', 'region_code', 'region_name', 'median_price_ons', 'median_earnings', 'affordability_ratio', 'imd_score_wtd']
